# Illinois — Chapter 215 ILCS (Insurance) → `data/illinois/ins_codes/*.md`

Illinois **insurance** law lives in **Chapter 215** of the **Illinois Compiled Statutes (ILCS)** (see **[ilga.gov ILCS](https://www.ilga.gov/legislation/ilcs/ilcs3.asp)**). Justia publishes the same material as a browse tree: **[Chapter 215 — Insurance](https://law.justia.com/codes/illinois/chapter-215/)** (`/codes/illinois/chapter-215/…`).

Unlike some other states on Justia, **many ILCS acts list “Article …” pages** where **an entire article is one HTML page**; smaller acts may put **all sections on the act landing page**. There is **not** always a per-section URL. This notebook **BFS**-crawls every index path under **`/codes/illinois/chapter-215/`**, then **splits** each page’s **`div.primary-content`** text on **`(215 ILCS <act>/<sec>)`** markers and writes **one Markdown file per ILCS subsection**.

**Cloudflare** often blocks plain **`httpx`**; we use **`curl_cffi`** with **`impersonate="chrome120"`** (same pattern as **`ins_ipynb/idaho.ipynb`**).

**Output files:** **`ILCS_sec_<act>_<sec>.md`** (e.g. `ILCS_sec_5_1.md` for **215 ILCS 5/1**). The header cites the **Justia mirror page** the text came from and the **215 ILCS** citation.

**Config:** **`CODE_YEAR`** seeds **`/codes/illinois/{year}/chapter-215/`** (normalized internally to the canonical **`/codes/illinois/chapter-215/…`** paths). **`MAX_SECTIONS`** caps **how many section files** are written (**0** = no cap). **`MAX_DISCOVERY_PAGES`** caps **HTML pages fetched** during the crawl (**0** = no cap). **`REUSE_DISCOVERED_URLS`** loads **`_illinois_chapter215_crawl_urls.txt`** and **re-fetches** those pages (skips BFS only). A **fresh crawl** parses each page **once** during the walk (no double-fetch).

**Politeness:** **`REQUEST_DELAY_SEC`** between requests.

Then run **`python -m app.ingest`** from the project root.


In [1]:
%pip install -q curl_cffi beautifulsoup4


You should consider upgrading via the '/Users/apps/Downloads/ZProjects/RAG/.venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [2]:
from __future__ import annotations

import re
import time
from collections import deque
from pathlib import Path
from urllib.parse import urljoin, urlparse

from bs4 import BeautifulSoup
from curl_cffi import requests as curl_requests

BASE = "https://law.justia.com"
PATH_PREFIX = "/codes/illinois/chapter-215"
CODE_YEAR = "2024"
TITLE_INDEX = f"{BASE}/codes/illinois/{CODE_YEAR}/chapter-215/"

OUT_DIR = Path("data") / "illinois" / "ins_codes"
OUT_DIR.mkdir(parents=True, exist_ok=True)

CURL_IMPERSONATE = "chrome120"
REQUEST_DELAY_SEC = 0.12
TIMEOUT = 90.0

MAX_SECTIONS = 0
MAX_DISCOVERY_PAGES = 0

SKIP_EXISTING = True

CRAWL_URL_LIST = OUT_DIR / "_illinois_chapter215_crawl_urls.txt"
REUSE_DISCOVERED_URLS = True

ILCS_MARKER = re.compile(r"\((\d+) ILCS (\d+)/(\d+[A-Za-z]?)\)")


In [3]:
def curl_get(url: str) -> str:
    time.sleep(REQUEST_DELAY_SEC)
    r = curl_requests.get(url, impersonate=CURL_IMPERSONATE, timeout=TIMEOUT)
    r.raise_for_status()
    return r.text


def path_key(u: str) -> str:
    return urlparse(u).path.rstrip("/")


def canon_path(p: str) -> str:
    """Map /codes/illinois/2024/chapter-215/… → /codes/illinois/chapter-215/…"""
    p = p.rstrip("/")
    m = re.match(r"^/codes/illinois/20\d\d(/chapter-215(?:/.*)?)$", p)
    if m:
        return "/codes/illinois" + m.group(1)
    return p


def under_chapter215(p: str) -> bool:
    c = canon_path(p)
    return c == PATH_PREFIX or c.startswith(PATH_PREFIX + "/")


def crawl_chapter215_collect() -> tuple[list[str], dict[tuple[str, str], tuple[str, str, str]]]:
    """BFS Chapter 215; each fetched page is parsed once into ILCS chunks (deduped by act/sec)."""
    start = TITLE_INDEX
    seen: set[str] = set()
    c0 = canon_path(path_key(start))
    in_q: set[str] = {c0}
    q: deque[str] = deque([start])
    ordered: list[str] = []
    by_key: dict[tuple[str, str], tuple[str, str, str]] = {}
    fetches = 0
    while q:
        if MAX_DISCOVERY_PAGES and fetches >= MAX_DISCOVERY_PAGES:
            break
        url = q.popleft()
        ck = canon_path(path_key(url))
        in_q.discard(ck)
        if ck in seen:
            continue
        seen.add(ck)
        fetch_url = BASE + ck + "/"
        page_url = fetch_url
        html = curl_get(fetch_url)
        fetches += 1
        ordered.append(page_url)
        raw = extract_primary_text(html)
        for act, sec, marker, chunk in split_ilcs_chunks(raw):
            k = (act, sec)
            if k not in by_key:
                by_key[k] = (marker, chunk, page_url)
        soup = BeautifulSoup(html, "html.parser")
        for a in soup.find_all("a", href=True):
            absu = urljoin(fetch_url, a["href"])
            p = canon_path(path_key(absu))
            if not under_chapter215(p):
                continue
            if "/section-" in p.lower():
                continue
            if p in seen or p in in_q:
                continue
            in_q.add(p)
            q.append(BASE + p + "/")
    return ordered, by_key


def extract_primary_text(html: str) -> str:
    soup = BeautifulSoup(html, "html.parser")
    pc = soup.select_one("div.primary-content")
    return pc.get_text("\n", strip=True) if pc else ""


def strip_justia_boilerplate(text: str) -> str:
    drop_prefixes = (
        "Go to Previous Versions",
        "View All Versions",
        "Learn more",
        "This media-neutral citation",
    )
    lines = text.split("\n")
    out: list[str] = []
    skip_until_substantive = True
    for line in lines:
        s = line.strip()
        if not s:
            if not skip_until_substantive:
                out.append("")
            continue
        if any(s.startswith(p) for p in drop_prefixes):
            continue
        if s.startswith("20") and "Illinois" in s and ("Comp" in s or "Compiled" in s):
            continue
        if s in {"Next", "Previous", "Universal Citation:"}:
            continue
        if s.startswith("Disclaimer:") or s.startswith("These codes may not"):
            continue
        skip_until_substantive = False
        out.append(s)
    return "\n".join(out).strip()


def split_ilcs_chunks(text: str) -> list[tuple[str, str, str, str]]:
    """Return [(act, sec, marker, chunk_text), ...] from primary text."""
    text = strip_justia_boilerplate(text)
    matches = list(ILCS_MARKER.finditer(text))
    rows: list[tuple[str, str, str, str]] = []
    for i, m in enumerate(matches):
        start = m.start()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        chunk = text[start:end].strip()
        act, sec = m.group(2), m.group(3)
        rows.append((act, sec, m.group(0), chunk))
    return rows


def act_sec_sort_key(act: str, sec: str) -> tuple:
    def part_key(p: str) -> tuple:
        out: list[tuple[int, int | str]] = []
        head, tail = "", p
        while tail and tail[0].isdigit():
            head += tail[0]
            tail = tail[1:]
        if head:
            out.append((0, int(head)))
        if tail:
            out.append((1, tail.lower()))
        return tuple(out) if out else ((1, p.lower()),)

    return (int(act),) + part_key(sec)


def act_sec_filename(act: str, sec: str) -> str:
    safe_sec = sec.replace("-", "_")
    return f"ILCS_sec_{act}_{safe_sec}.md"


def download_chapter215() -> dict[str, int]:
    if REUSE_DISCOVERED_URLS and CRAWL_URL_LIST.exists() and CRAWL_URL_LIST.stat().st_size > 50:
        crawl_urls = [ln.strip() for ln in CRAWL_URL_LIST.read_text(encoding="utf-8").splitlines() if ln.strip()]
        print(f"Loaded {len(crawl_urls)} crawl URLs from {CRAWL_URL_LIST.name} (re-fetching for ILCS split)")
        by_key: dict[tuple[str, str], tuple[str, str, str]] = {}
        for page_url in crawl_urls:
            html = curl_get(page_url)
            raw = extract_primary_text(html)
            for act, sec, marker, chunk in split_ilcs_chunks(raw):
                k = (act, sec)
                if k not in by_key:
                    by_key[k] = (marker, chunk, page_url)
    else:
        crawl_urls, by_key = crawl_chapter215_collect()
        print(
            f"Crawled {len(crawl_urls)} Chapter 215 pages; collected {len(by_key)} unique ILCS section chunks"
        )
        CRAWL_URL_LIST.write_text("\n".join(crawl_urls) + "\n", encoding="utf-8")

    rows = sorted(
        (
            (act, sec, marker, chunk, page_url, act_sec_sort_key(act, sec))
            for (act, sec), (marker, chunk, page_url) in by_key.items()
        ),
        key=lambda r: r[5],
    )

    if MAX_SECTIONS:
        rows = rows[:MAX_SECTIONS]
        print(f"Limited export to first {len(rows)} ILCS chunks (MAX_SECTIONS)")

    wrote = skipped = failed = 0
    for i, (act, sec, marker, chunk, page_url, _sk) in enumerate(rows, 1):
        dest = OUT_DIR / act_sec_filename(act, sec)
        ilcs = f"215 ILCS {act}/{sec}"
        if SKIP_EXISTING and dest.exists() and dest.stat().st_size > 80:
            skipped += 1
        else:
            try:
                title = f"Illinois Compiled Statutes — {ilcs}"
                md = (
                    f"# {title}\n\n"
                    f"**Illinois Compiled Statutes — Chapter 215 (Insurance-related ILCS)**\n\n"
                    f"**ILCS citation:** {ilcs}\n\n"
                    f"**Source (Justia mirror page):** {page_url}\n\n"
                    f"**Verify / browse official ILCS:** [ilga.gov — ILCS](https://www.ilga.gov/legislation/ilcs/ilcs3.asp)\n\n"
                    f"**Marker:** {marker}\n\n"
                    f"---\n\n"
                    f"{chunk}\n"
                )
                dest.write_text(md, encoding="utf-8")
                wrote += 1
            except Exception as e:
                print(f"FAIL {ilcs}: {e}")
                failed += 1
        if i % 200 == 0:
            print(f"… {i}/{len(rows)} (wrote={wrote} skipped={skipped} failed={failed})")

    print(f"Done. wrote={wrote} skipped={skipped} failed={failed} → {OUT_DIR.resolve()}")
    return {"wrote": wrote, "skipped": skipped, "failed": failed}


download_chapter215()


Crawled 118 Chapter 215 pages; collected 1195 unique ILCS section chunks
… 200/1195 (wrote=200 skipped=0 failed=0)
… 400/1195 (wrote=400 skipped=0 failed=0)
… 600/1195 (wrote=600 skipped=0 failed=0)
… 800/1195 (wrote=800 skipped=0 failed=0)
… 1000/1195 (wrote=1000 skipped=0 failed=0)
Done. wrote=1195 skipped=0 failed=0 → /Users/apps/Downloads/ZProjects/RAG/ins_ipynb/data/illinois/ins_codes


{'wrote': 1195, 'skipped': 0, 'failed': 0}

## Next step

`python -m app.ingest` from the project root.
